# 03 — Expérimentations finales avec MLflow

Ce notebook est volontairement plus simple que `02_modelisation.ipynb`.

Le travail principal de comparaison et d'analyse des modèles est réalisé dans le notebook `02`.
Ici, MLflow sert surtout à :

- enregistrer les runs finaux ;
- conserver les principaux paramètres ;
- comparer MAE, RMSE, R² test et R² moyen en validation croisée ;
- sauvegarder les pipelines entraînés ;
- conserver le modèle champion dans `models/best_model.joblib`.

Modèles retenus :

1. `LinearRegression`
2. `RidgeCV`
3. `Lasso(alpha=1)`
4. `ElasticNetCV`
5. `XGBoost`

## 1. Imports et configuration MLflow

In [12]:
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNetCV, Lasso, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBRegressor

print("Version MLflow :", mlflow.__version__)

Version MLflow : 3.15.2


In [13]:
experiment = mlflow.set_experiment("prediction_prix_automobile_final")

print("Nom :", experiment.name)
print("ID :", experiment.experiment_id)

Nom : prediction_prix_automobile_final
ID : 1


## 2. Chargement des données

In [14]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

X = df.drop(columns=["PRICE"])
y = df["PRICE"]

print("Dataset :", df.shape)
print("X :", X.shape)
print("y :", y.shape)

Dataset : (1748, 12)
X : (1748, 11)
y : (1748,)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (1398, 11)
X_test  : (350, 11)
y_train : (1398,)
y_test  : (350,)


## 3. Prétraitement

Deux prétraitements sont utilisés :

- `preprocessor` : variables numériques conservées telles quelles ;
- `preprocessor_scaled` : variables numériques standardisées pour Ridge, Lasso et ElasticNet.

In [16]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            "passthrough",
            variables_numeriques
        )
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            StandardScaler(),
            variables_numeriques
        )
    ]
)

## 4. Modèles finaux

Les hyperparamètres reprennent les choix retenus dans `02_modelisation.ipynb`.

In [18]:
modeles = {
    "LinearRegression": {
        "pipeline": Pipeline([
            ("preparation", preprocessor),
            ("modele", LinearRegression())
        ]),
        "params": {
            "model_name": "LinearRegression"
        }
    },

    "RidgeCV": {
        "pipeline": Pipeline([
            ("preparation", preprocessor_scaled),
            ("modele", RidgeCV(
                alphas=[0.01, 0.1, 1, 10, 100],
                cv=5,
                scoring="r2"
            ))
        ]),
        "params": {
            "model_name": "RidgeCV",
            "alphas": "0.01,0.1,1,10,100",
            "cv_interne": 5
        }
    },

    "Lasso_alpha_1": {
        "pipeline": Pipeline([
            ("preparation", preprocessor_scaled),
            ("modele", Lasso(
                alpha=1,
                max_iter=10000
            ))
        ]),
        "params": {
            "model_name": "Lasso",
            "alpha": 1,
            "max_iter": 10000
        }
    },

    "ElasticNetCV": {
        "pipeline": Pipeline([
            ("preparation", preprocessor_scaled),
            ("modele", ElasticNetCV(
                alphas=[0.001, 0.01, 0.1, 1, 10],
                l1_ratio=[0.1, 0.5, 0.9],
                cv=5,
                max_iter=10000
            ))
        ]),
        "params": {
            "model_name": "ElasticNetCV",
            "alphas": "0.001,0.01,0.1,1,10",
            "l1_ratio": "0.1,0.5,0.9",
            "cv_interne": 5,
            "max_iter": 10000
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("preparation", preprocessor),
            ("modele", XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=3,
                random_state=42,
                objective="reg:squarederror"
            ))
        ]),
        "params": {
            "model_name": "XGBoost",
            "n_estimators": 100,
            "learning_rate": 0.1,
            "max_depth": 3,
            "model_random_state": 42,
            "objective": "reg:squarederror"
        }
    }
}

print("Modèles à exécuter :", list(modeles.keys()))

Modèles à exécuter : ['LinearRegression', 'RidgeCV', 'Lasso_alpha_1', 'ElasticNetCV', 'XGBoost']


## 5. Exécution des runs MLflow

Tous les modèles utilisent le même jeu d'entraînement/test, les mêmes métriques et une validation croisée externe à 5 plis.

In [19]:
resultats = []
modeles_entraines = {}

for run_name, config in modeles.items():

    pipeline = config["pipeline"]

    with mlflow.start_run(run_name=f"final_{run_name.lower()}"):

        # Entraînement
        pipeline.fit(X_train, y_train)

        # Prédictions sur le jeu de test
        y_pred = pipeline.predict(X_test)

        # Métriques
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2_test = r2_score(y_test, y_pred)

        # Validation croisée externe
        scores_cv = cross_val_score(
            pipeline,
            X,
            y,
            cv=5,
            scoring="r2"
        )
        r2_cv_mean = scores_cv.mean()

        # Paramètres communs
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("random_state_split", 42)
        mlflow.log_param("dataset_rows", len(df))

        # Paramètres du modèle
        for param_name, param_value in config["params"].items():
            mlflow.log_param(param_name, param_value)

        # Paramètres choisis automatiquement
        modele_fit = pipeline.named_steps["modele"]

        if hasattr(modele_fit, "alpha_"):
            mlflow.log_param(
                "selected_alpha",
                float(modele_fit.alpha_)
            )

        if hasattr(modele_fit, "l1_ratio_"):
            mlflow.log_param(
                "selected_l1_ratio",
                float(modele_fit.l1_ratio_)
            )

        # Métriques MLflow
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2_test", r2_test)
        mlflow.log_metric("r2_cv_mean", r2_cv_mean)

        # Pipeline complet
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            serialization_format="cloudpickle"
        )
        # Résultats locaux
        resultats.append({
            "Modèle": run_name,
            "MAE": mae,
            "RMSE": rmse,
            "R² test": r2_test,
            "R² CV moyen": r2_cv_mean
        })

        modeles_entraines[run_name] = pipeline

        print(
            run_name,
            "| MAE:", round(mae, 2),
            "| RMSE:", round(rmse, 2),
            "| R² test:", round(r2_test, 3),
            "| R² CV:", round(r2_cv_mean, 3)
        )

2026/10/08 22:41:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression | MAE: 2265.69 | RMSE: 4249.43 | R² test: 0.702 | R² CV: 0.722


2026/10/08 22:42:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RidgeCV | MAE: 1806.89 | RMSE: 3755.12 | R² test: 0.768 | R² CV: 0.764


2026/10/08 22:42:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso_alpha_1 | MAE: 1699.98 | RMSE: 3817.56 | R² test: 0.76 | R² CV: 0.756


2026/10/08 22:42:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ElasticNetCV | MAE: 1776.38 | RMSE: 3730.02 | R² test: 0.771 | R² CV: 0.772


2026/10/08 22:42:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost | MAE: 2049.31 | RMSE: 4433.8 | R² test: 0.676 | R² CV: 0.773


## 6. Comparaison finale

In [20]:
resultats_mlflow = pd.DataFrame(resultats)

resultats_mlflow = (
    resultats_mlflow
    .sort_values(
        by=["R² test", "RMSE"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

resultats_mlflow.round(3)

,Modèle,MAE,RMSE,R² test,R² CV moyen
0,ElasticNetCV,1776.378,3730.021,0.771,0.772
1,RidgeCV,1806.894,3755.120,0.768,0.764
2,Lasso_alpha_1,1699.975,3817.561,0.760,0.756
3,LinearRegression,2265.692,4249.428,0.702,0.722
4,XGBoost,2049.314,4433.795,0.676,0.773


## 7. Modèle champion

La sélection principale a été réalisée dans `02_modelisation.ipynb`.

`ElasticNetCV` est retenu comme modèle champion pour son meilleur compromis global entre MAE, RMSE, R² test et stabilité en validation croisée.

Le pipeline complet est sauvegardé dans `models/best_model.joblib`.

In [21]:
champion_name = "ElasticNetCV"
champion = modeles_entraines[champion_name]

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

champion_path = models_dir / "best_model.joblib"

joblib.dump(
    champion,
    champion_path
)

print("Champion :", champion_name)
print("Modèle sauvegardé :", champion_path)

Champion : ElasticNetCV
Modèle sauvegardé : ../models/best_model.joblib


## Conclusion

MLflow sert ici à enregistrer et comparer les cinq modèles finaux retenus dans `02_modelisation.ipynb`.

`ElasticNetCV` est conservé comme modèle champion et sauvegardé dans `models/best_model.joblib`.